In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import (
    load_inventory,
    load_shipment,
    load_orders,
    load_order_items,
    load_sales,
    load_promotions,
    load_products,
)

import matplotlib.pyplot as plt
import pandas as pd

DATA_ROOT = project_root / "data/datathon-2026-round-1"
inventory = load_inventory(DATA_ROOT)
shipment = load_shipment(DATA_ROOT)
orders = load_orders(DATA_ROOT)
order_items = load_order_items(DATA_ROOT)
sales = load_sales(DATA_ROOT)
promotions = load_promotions(DATA_ROOT)
products = load_products(DATA_ROOT)

In [ ]:
# list all unique order_status in orders
order_statuses = orders["order_status"].unique()
print("Unique order_status values in orders:")
print(order_statuses)
# count order_status na
order_status_na_count = orders["order_status"].isna().sum()
print(f"Number of orders with missing order_status: {order_status_na_count}")
# check if there are any order with order_id in orders but not in shipment
orders_with_shipment = orders[orders["order_id"].isin(shipment["order_id"])]
orders_without_shipment = orders[~orders["order_id"].isin(shipment["order_id"])]
print(f"Number of orders with shipment: {len(orders_with_shipment)}")

Có 1708 ngày revenue lệch so với sales nếu tính tất cả trạng thái đơn hàng

Nếu loại trừ cancel và returned không tính vào revenue của ngày đó thì có hơn 2000 ngày lệch.

Chưa biết nguyên nhân gây lệch ở 1708 ngày.

In [ ]:
# Compare daily revenue from delivered orders vs sales revenue.
order_items_rev = order_items.copy()
order_items_rev["product_revenue"] = (
    order_items_rev["unit_price"] * order_items_rev["quantity"] - order_items_rev["discount_amount"]
 )
order_rev_by_order = order_items_rev.groupby("order_id", as_index=False)["product_revenue"].sum()
orders_with_shipment = orders[orders["order_id"].isin(shipment["order_id"])].copy()
orders_delivered = orders_with_shipment[orders_with_shipment["order_status"] == "delivered"].copy()
orders_delivered_ids = orders_delivered[["order_id"]].drop_duplicates()
shipment_dates = shipment[["order_id", "delivery_date"]].copy()
shipment_dates["delivery_date"] = pd.to_datetime(shipment_dates["delivery_date"]).dt.date
shipment_dates = shipment_dates.dropna(subset=["delivery_date"])
shipment_dates = shipment_dates.groupby("order_id", as_index=False)["delivery_date"].max()
order_rev_by_order = order_rev_by_order.merge(orders_delivered_ids, on="order_id", how="inner")
order_rev_by_order = order_rev_by_order.merge(shipment_dates, on="order_id", how="inner")
order_rev_by_date = order_rev_by_order.groupby("delivery_date", as_index=False)["product_revenue"].sum()
order_rev_by_date.rename(columns={"delivery_date": "date", "product_revenue": "revenue_orders"}, inplace=True)
sales_rev_by_date = sales[["date", "Revenue"]].copy()
sales_rev_by_date["date"] = pd.to_datetime(sales_rev_by_date["date"]).dt.date
sales_rev_by_date.rename(columns={"Revenue": "revenue_sales"}, inplace=True)
revenue_compare = order_rev_by_date.merge(sales_rev_by_date, on="date", how="left")
revenue_compare["revenue_diff"] = revenue_compare["revenue_sales"] - revenue_compare["revenue_orders"]
revenue_compare["revenue_ratio"] = revenue_compare["revenue_sales"] / revenue_compare["revenue_orders"]
# sum revenue_diff
total_revenue_diff = revenue_compare["revenue_diff"].sum()
print(f"Total revenue difference (sales - orders): {total_revenue_diff:.2f}")
# print days where abs revenue_diff >= 0.1
diff_days = revenue_compare[revenue_compare["revenue_diff"].abs() >= 0.1]

print("Days with revenue difference:")
print(diff_days[["date", "revenue_orders", "revenue_sales", "revenue_diff", "revenue_ratio"]])  


In [ ]:
# Diagnose revenue differences.
orders_only = revenue_compare[revenue_compare["revenue_sales"].isna()].copy()
sales_only = revenue_compare[revenue_compare["revenue_orders"].isna()].copy()
both = revenue_compare.dropna(subset=["revenue_orders", "revenue_sales"]).copy()

print("Days only in orders:", len(orders_only))
print("Days only in sales:", len(sales_only))
print("Days in both:", len(both))

print("Total revenue_orders:", both["revenue_orders"].sum())
print("Total revenue_sales:", both["revenue_sales"].sum())
print("Total diff on overlap:", (both["revenue_sales"] - both["revenue_orders"]).sum())

# Show top 10 absolute differences on overlapping dates.
top_diff = both.reindex(both["revenue_diff"].abs().sort_values(ascending=False).index)
top_diff.head(10)[["date", "revenue_orders", "revenue_sales", "revenue_diff", "revenue_ratio"]]

In [ ]:
# List returned/canceled orders by day.
orders_rc = orders[orders["order_status"].isin(["returned", "canceled"])].copy()
orders_rc["order_date"] = pd.to_datetime(orders_rc["order_date"]).dt.date
orders_rc_by_day = (
    orders_rc.groupby(["order_date", "order_status"])
    .size()
    .reset_index(name="order_count")
    .sort_values(["order_date", "order_status"])
 )
orders_rc_by_day.head(20)

In [ ]:
grouped = order_items.groupby("product_id")["unit_price"].mean().reset_index()
merged = pd.merge(grouped, products[["product_id", "price"]], on="product_id", how="inner")
merged.head()


In [ ]:
# promotions: promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
# order_items: order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
# orders: order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source

# Compute order-level revenue and profit (shipping fee is per order).
order_items["product_revenue"] = (
    order_items["unit_price"] * order_items["quantity"] - order_items["discount_amount"]
 )
order_revenue = order_items.groupby("order_id", as_index=False)["product_revenue"].sum()
shipping_fee_by_order = shipment.groupby("order_id", as_index=False)["shipping_fee"].sum()
orders_delivered = orders[orders["order_status"] == "delivered"].copy()
orders_dates = orders_delivered[["order_id", "date"]].copy()
orders_dates["date"] = pd.to_datetime(orders_dates["date"]).dt.date
order_revenue = order_revenue.merge(orders_dates, on="order_id", how="inner")
order_revenue = order_revenue.merge(shipping_fee_by_order, on="order_id", how="left")
order_revenue["shipping_fee"] = order_revenue["shipping_fee"].fillna(0)
order_revenue["profit"] = order_revenue["product_revenue"] - order_revenue["shipping_fee"]
order_revenue.head()

In [ ]:
profit_by_date = order_revenue.groupby("date", as_index=False)["profit"].sum()
profit_by_date.rename(columns={"order_date": "date"}, inplace=True)
sales["date"] = pd.to_datetime(sales["date"]).dt.date
sales["profit_sales"] = sales["Revenue"] - sales["COGS"]
profit_by_date = profit_by_date.merge(sales[["date", "profit_sales"]], on="date", how="left")
profit_by_date